# Packaging and Registering a Custom Model

This walkthrough shows how to bundle your own model package, upload its wheel,
register a configuration with dependency metadata, and invoke the model via
the Model Zoo registry API.

## 1. Create a lightweight package

Start with a minimal Python package that exposes a callable. In this example
`myteam_marketing_mix` defines a `train_and_infer` function that matches the
library runner contract (inputs, params, context).

In [ ]:
%%bash
set -euxo pipefail
mkdir -p demo_src/myteam_marketing_mix
cat <<'PY' > demo_src/myteam_marketing_mix/__init__.py
def train_and_infer(inputs, params, context):
    # simple example: scale feature and return diagnostics
    feature = inputs['X']
    scaled = [v * params.get('scale', 1.0) for v in feature]
    return {
        'predictions': scaled,
        'diagnostics': {'mean': sum(scaled) / len(scaled)}
    }
PY

cat <<'TOML' > demo_src/pyproject.toml
[build-system]
requires = ['setuptools>=68', 'wheel']
build-backend = 'setuptools.build_meta'

[project]
name = 'myteam_marketing_mix'
version = '1.2.0'
description = 'Demo marketing mix model for Model Zoo'
dependencies = ['pandas>=2.2']
TOML

python -m build --wheel --outdir dist demo_src


The wheel in `dist/` can be uploaded to your package index or through the
Model Zoo registry UI. Record the artifact name so the configuration can
declare it as a dependency.

In [ ]:
import json
from textwrap import dedent

config_payload = {
    'model': {
        'id': 'marketing_mix_v1',
        'name': 'Marketing Mix (Demo)',
        'version': '1.0.0',
        'kind': 'library',
        'binding': {
            'module': 'myteam_marketing_mix',
            'callable': 'train_and_infer'
        },
        'inputs': ['X']
    },
    'dependencies': [
        'myteam_marketing_mix==1.2.0',
        {'env': 'prod', 'packages': ['prophet>=1.1']}
    ],
    'hyperparameters': {
        'schema': [
            {'name': 'scale', 'dtype': 'float', 'default': 1.0}
        ],
        'values': [
            {'model_id': 'marketing_mix_v1', 'param': 'scale', 'value': '1.0', 'env': 'dev'}
        ]
    },
    'variables': {
        'definitions': [
            {'name': 'X', 'source': 'spend_series'}
        ]
    },
    'context': {'timeout_s': 120}
}

print(dedent(json.dumps(config_payload, indent=2)))


In [ ]:
import requests

response = requests.post(
    'http://localhost:8000/registry/configs',
    json={'env': 'prod', 'config': config_payload}
)
response.raise_for_status()
response.json()


The response contains the stored spec along with dependency metadata. Once
registered, runs can target `marketing_mix_v1` and the runtime will install
`myteam_marketing_mix==1.2.0` plus the prod-only extras before importing
`train_and_infer`.